# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. `[1. 2. 3.] [0 1 1]`. Обычное деление `/` всегда даёт float, а `//` берёт целую часть: 2 // 4 = 0, 4 // 4 = 1, 6 // 4 = 1.
2. `[4. 4. 4.]`. `axis=1` складывает каждую строку, в строке четыре единицы, строк три. `np.ones` создаёт float, поэтому с точками.
3. `[[0 1 2] [1 2 3] [2 3 4]]`. Строка формы (3,) и столбец формы (3, 1) растягиваются до 3 на 3, получается таблица сложения i + j.
4. `6.0`. Корни из 1, 4, 9 это 1, 2, 3, в сумме 6, а `np.sqrt` возвращает float.
5. `7.5`. Квадраты 1, 4, 9, 16 в сумме дают 30, делим на 4.

In [1]:
# проверка предсказаний
import numpy as np

a = np.array([2, 4, 6]); print(a / 2, a // 4)
m = np.ones((3, 4)); print(m.sum(axis=1))
print(np.arange(3) + np.arange(3).reshape(3, 1))
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [4]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [5]:
# 1. выручка по чистым данным
print("строк в clean:", clean.sum())

months = np.array(["2026-09", "2026-10"])
rev = np.array([amount[clean & (month == m)].sum() for m in months])
change_pct = (rev[1] / rev[0] - 1) * 100
print(f"сентябрь: {rev[0]:,.0f} ₸, октябрь: {rev[1]:,.0f} ₸, изменение {change_pct:.1f}%")

строк в clean: 4739
сентябрь: 35,243,410 ₸, октябрь: 34,419,520 ₸, изменение -2.3%


**Ответ:** в чистых данных 4 739 строк. Выручка 35 243 410 ₸ в сентябре и 34 419 520 ₸ в октябре, то есть падение на 2,3%. С ДЗ 2 совпадает до тенге: там после удаления дублей и TEST получились те же суммы, а по сырым данным было -22,7%.

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [6]:
# 2. таблица категория x месяц
cats = np.unique(cat)
cat_table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                      for c in cats])
for c, r in zip(cats, cat_table):
    print(f"{c:<12} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(cat_table.shape)

total = cat_table.sum(axis=1)   # по строкам: каждая категория за два месяца
print(f"\nлидер за два месяца: {cats[total.argmax()]}, {total.max():,.0f} ₸")

cat_change = (cat_table[:, 1] / cat_table[:, 0] - 1) * 100
for c, ch in zip(cats, cat_change):
    print(f"{c:<12} {ch:+.1f}%")
print("сильнее всех упала:", cats[cat_change.argmin()])

Дом             4,997,730    5,125,790
Книги           1,065,300    1,140,910
Красота         2,610,950    2,678,990
Одежда          9,167,100    8,505,730
Электроника    17,402,330   16,968,100
(5, 2)

лидер за два месяца: Электроника, 34,370,430 ₸
Дом          +2.6%
Книги        +7.1%
Красота      +2.6%
Одежда       -7.2%
Электроника  -2.5%
сильнее всех упала: Одежда


**Ответ:** больше всего приносит Электроника, 34,4 млн ₸ за два месяца, это почти половина всей выручки. Сильнее всех упала Одежда, на 7,2%. Электроника тоже просела на 2,5%, а Дом, Книги и Красота наоборот немного выросли.

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [7]:
# 3. число заказов: сумма маски, а не сумма amount
cat_count = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                      for c in cats])

avg_check = cat_table / cat_count   # одна операция над двумя таблицами 5 x 2
for c, n, a in zip(cats, cat_count, avg_check):
    print(f"{c:<12} заказов {n[0]:>4} {n[1]:>4}   средний чек {a[0]:>8,.0f} {a[1]:>8,.0f}")

best = avg_check[:, 1].argmax()
print(f"\nсамый высокий чек в октябре: {cats[best]}, {avg_check[best, 1]:,.0f} ₸")

Дом          заказов  476  476   средний чек   10,499   10,768
Книги        заказов  265  299   средний чек    4,020    3,816
Красота      заказов  422  435   средний чек    6,187    6,159
Одежда       заказов  678  652   средний чек   13,521   13,046
Электроника  заказов  531  505   средний чек   32,773   33,600

самый высокий чек в октябре: Электроника, 33,600 ₸


**Ответ:** самый высокий средний чек в октябре у Электроники, 33 600 ₸. Это в 2,6 раза больше, чем у Одежды (13 046 ₸), и почти в 9 раз больше, чем у Книг (3 816 ₸).

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [8]:
# 4. таблица город x месяц, как на уроке
cities = np.unique(city)
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])

# (6, 2) делим на (2,): строка с итогами месяцев растягивается на все города
share = table / table.sum(axis=0)
print("сумма долей по столбцам:", share.sum(axis=0))
for c, s in zip(cities, share):
    print(f"{c:<10} {s[0]:>6.1%} {s[1]:>6.1%}")

сумма долей по столбцам: [1. 1.]
Актобе       7.3%   7.4%
Алматы      41.1%  42.5%
Астана      25.6%  25.7%
Атырау       2.6%   2.8%
Караганда   10.4%   9.8%
Шымкент     12.9%  11.7%


**Ответ:** каждый столбец в сумме даёт 1. Доля Шымкента в выручке месяца упала с 12,9% до 11,7%, а у Алматы наоборот выросла с 41,1% до 42,5%.

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [9]:
# 5. вклад каждого города в общее падение
change = table[:, 1] - table[:, 0]
print(f"общее изменение: {change.sum():,.0f} ₸")

contrib = change / change.sum()
for c, ch, k in zip(cities, change, contrib):
    print(f"{c:<10} {ch:>+10,.0f} ₸  {k:+.1%}")

общее изменение: -823,890 ₸
Актобе        -36,540 ₸  +4.4%
Алматы       +141,950 ₸  -17.2%
Астана       -183,690 ₸  +22.3%
Атырау        +45,870 ₸  -5.6%
Караганда    -276,840 ₸  +33.6%
Шымкент      -514,640 ₸  +62.5%


**Ответ:** выручка упала на 823 890 ₸, и 62,5% этого падения дал Шымкент, ещё 33,6% Караганда. Алматы и Атырау выросли, поэтому у них вклад с минусом.

**Менеджеру:** причину падения стоит искать в Шымкенте, там же самая медленная доставка (только 30,7% заказов за 1-2 дня в ДЗ 3) и больше всего отмен (20,4% в ДЗ 2).

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [10]:
# 6. table.sum(axis=1) имеет форму (6,), а table (6, 2).
# формы сравниваются справа: 2 и 6 не равны и ни одна из них не 1, поэтому ошибка
try:
    share = table / table.sum(axis=1)
except ValueError as e:
    print("ошибка:", e)

# keepdims=True оставляет форму (6, 1), и столбец итогов растягивается вправо
city_share = table / table.sum(axis=1, keepdims=True)
print("сумма по строкам:", city_share.sum(axis=1))
for c, s in zip(cities, city_share):
    print(f"{c:<10} сентябрь {s[0]:.1%}  октябрь {s[1]:.1%}")

ошибка: operands could not be broadcast together with shapes (6,2) (6,) 
сумма по строкам: [1. 1. 1. 1. 1. 1.]
Актобе     сентябрь 50.4%  октябрь 49.6%
Алматы     сентябрь 49.8%  октябрь 50.2%
Астана     сентябрь 50.5%  октябрь 49.5%
Атырау     сентябрь 48.8%  октябрь 51.2%
Караганда  сентябрь 52.0%  октябрь 48.0%
Шымкент    сентябрь 53.0%  октябрь 47.0%


**Ответ:** код падает, потому что итоги по строкам имеют форму (6,), а таблица (6, 2), и NumPy не может их совместить. С `keepdims=True` всё работает. Сильнее всего перекос у Шымкента: на сентябрь приходится 53% его выручки за два месяца, на октябрь только 47%.

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [11]:
# 7. z-оценка суммы заказа (clean уже содержит только оплаченные)
x = amount[clean]
z = (x - x.mean()) / x.std()
outliers = np.abs(z) > 3

print(f"среднее {x.mean():,.0f} ₸, std {x.std():,.0f} ₸")
print(f"заказов с |z| > 3: {outliers.sum()} из {len(x)}, доля {outliers.mean():.1%}")
print(f"самый дешёвый из них: {x[outliers].min():,.0f} ₸")
print(f"проверка: среднее z = {z.mean():.6f}, std z = {z.std():.6f}")

среднее 14,700 ₸, std 13,292 ₸
заказов с |z| > 3: 87 из 4739, доля 1.8%
самый дешёвый из них: 54,740 ₸
проверка: среднее z = 0.000000, std z = 1.000000


**Ответ:** 87 заказов (1,8%) отличаются от среднего больше чем на 3 стандартных отклонения. Все они дорогие, от 54 740 ₸ и выше, слишком дешёвых выбросов нет. У `z` среднее почти 0, а std ровно 1, как и должно быть после стандартизации.

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [12]:
# 8. два признака: сумма и дни доставки, чистые заказы без пропусков в days
m8 = clean & ~np.isnan(days)
X = np.column_stack([amount[m8], days[m8]])
np.set_printoptions(suppress=True)   # чтобы 139040 не печаталось как 1.3904e+05
print("форма X:", X.shape)
print("до:    min", X.min(axis=0), "  max", X.max(axis=0))

X_scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print("после: min", X_scaled.min(axis=0), "  max", X_scaled.max(axis=0))

форма X: (4595, 2)
до:    min [720.   1.]   max [139040.      8.]
после: min [0. 0.]   max [1. 1.]


**Ответ:** матрица `X` получилась 4 595 на 2. Сумма заказа идёт от 720 до 139 040 ₸, дни доставки от 1 до 8, после масштабирования оба столбца лежат от 0 до 1.

Зачем это модели: многие модели сравнивают объекты по расстоянию или подбирают веса градиентным спуском. Если один признак в тысячи раз больше другого, он просто задавит второй, и разница в 1 000 ₸ покажется модели важнее, чем разница в 7 дней доставки. Когда признаки в одном масштабе, каждый влияет на результат честно, а обучение сходится быстрее.